# Bear Blog Trending Feed: Data Collection and Analysis
**Mia Rosner — INFO 4617**

Planned collection: once daily at 8:00 p.m. Mountain Time for 14 days. Run the collection cell once at each scheduled time.

In [ ]:
import requests, feedparser
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt

FEED_URL = 'https://bearblog.dev/discover/feed/'
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)


## Collect one daily observation
Run this cell at 8 p.m. Mountain Time. It saves one CSV per collection date.

In [ ]:
collected_at = datetime.now(timezone.utc)
response = requests.get(FEED_URL, headers={'User-Agent': 'INFO4617-BearBlogResearch/1.0 (educational project)'}, timeout=20)
response.raise_for_status()
feed = feedparser.parse(response.content)
records = []
for entry in feed.entries:
    published = entry.get('published_parsed')
    if published is None:
        continue
    published_at = datetime(*published[:6], tzinfo=timezone.utc)
    records.append({'post_url': entry.get('link'), 'published_at': published_at.isoformat(), 'collected_at': collected_at.isoformat(), 'age_hours': (collected_at-published_at).total_seconds()/3600})
daily = pd.DataFrame(records)
path = DATA_DIR / f"bearblog_{collected_at.strftime('%Y-%m-%d')}.csv"
daily.to_csv(path, index=False)
print('HTTP status:', response.status_code, '| Records:', len(daily), '| Saved:', path)
daily.head()


## Combine collections and analyze
Run after multiple daily CSV files have been collected.

In [ ]:
files = sorted(DATA_DIR.glob('bearblog_*.csv'))
if not files:
    raise FileNotFoundError('No daily CSV files found in data/.')
posts = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
posts['collected_at'] = pd.to_datetime(posts['collected_at'], utc=True)
summary = posts.groupby(posts['collected_at'].dt.date).agg(mean_age=('age_hours','mean'), median_age=('age_hours','median'), youngest=('age_hours','min'), oldest=('age_hours','max'), count=('post_url','count'))
display(summary)
for hours in [12, 24, 48]:
    print(f"Within {hours} hours: {(posts['age_hours'] <= hours).mean()*100:.1f}%")


In [ ]:
posts['age_hours'].plot(kind='hist', bins=10, edgecolor='black')
plt.xlabel('Post age (hours)'); plt.ylabel('Daily post observations')
plt.title('Age of Posts in Bear Blog Trending Feed'); plt.tight_layout(); plt.show()
summary['median_age'].plot(kind='line', marker='o')
plt.xlabel('Collection date'); plt.ylabel('Median post age (hours)')
plt.title('Daily Median Age of Featured Posts'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
